# Minimal OpenAI Helper Notebook

This notebook keeps a small reusable OpenAI helper flow for quick experiments before running the fuller RAG notebooks.

## 1. Load Environment and Create OpenAI Client

Load `.env`, validate that `OPENAI_API_KEY` is available, and initialize the OpenAI client.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv(".env")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY was not loaded. Check that .env is in the notebook's working directory.")

openai_client = OpenAI()

## 2. Define a Small LLM Helper Function

Define a typed helper function around the Responses API so later experiments can call `llm(prompt)`.

In [2]:
def llm(prompt: str) -> str:
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt,
    )
    return response.output_text

## 3. Run a Simple Test Prompt

Call the helper once to confirm the notebook setup is working end to end.

In [3]:
llm("Hey, what's up?")

'Hey! Not much — I’m here and ready to help. What’s on your mind?'

In [4]:

from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)


In [5]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [6]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your operating system:\n   - macOS: download the `.pkg` and install it\n   - Windows: download the `.msi` and install it\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and run:\n   ```bash\n   ollama run llama3\n   ```\n   This will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\n3. To test the local Ollama server, run:\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. If you want to use it from Python, install the client:\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": your_prompt}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```'

In [9]:
assistant.rag("How do I run Olama locally?")

'The context doesn’t include instructions for running **Olama locally**.\n\nIf you meant **MCP Inspector**, the command is:\n\n```bash\nnpx @modelcontextprotocol/inspector\n```\n\nIf you want, I can help with another question from the course FAQ.'

In [10]:
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

In [12]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Possibly, yes — but it depends on the course’s enrollment rules and whether it’s still open.\n\nTo find out quickly:\n1. Check the course page for **“enroll,” “join,” “sign up,”** or **deadlines**.\n2. See whether it’s **self-paced**, **open enrollment**, or **cohort-based**.\n3. If there’s an instructor, coordinator, or support contact, message them asking:\n   - whether late enrollment is allowed\n   - if you’ve missed any required start dates\n   - whether you’ll need special access or approval\n\nIf you want, send me the course name or the platform, and I can help you figure out the best next step or draft a message to ask.'

In [13]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [14]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [15]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration"}', call_id='call_holXdawTLJNqMkWjVUjeRHT6', name='search', type='function_call', id='fc_0cdaf9407b6fb6ad006a29c0055944819ebf3609b2e74607da', namespace=None, status='completed')]

In [16]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [17]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [18]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes, you can still join.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [19]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(651, 29)

In [20]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [21]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [22]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [23]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment register late join FAQ"}
function_call: search {"query":"course access enrollment open signup join after start FAQ"}


In [24]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open. If you’re just learning, you can start anytime.

If you’d like, I can also help with:
- whether you need to register first
- how certificates work
- what to do if you joined late

Anything else you want to explore?


In [25]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [26]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run Ollama locally install run"}
function_call: search {"query":"Ollama local setup run model localhost FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: download and install the `.pkg`
     - **Windows**: download and install the `.msi`
     - **Linux**: run:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a model locally**
   - In a terminal, run:
     ```bash
     ollama run llama3
     ```
   - This will download the model, start it locally, and open a chat interface.

3. **Check the local server**
   - Ollama runs a server on `localhost:11434`. Test it with:
     ```bash
     curl http://localhost:11434
     ```
   - You should get a JSON response.

4. **Use it from Python**
   - Install the client:
     ```bash
     pip install ollama
     ```
   - Example:
     ```python


'To run Ollama locally:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: download and install the `.pkg`\n     - **Windows**: download and install the `.msi`\n     - **Linux**: run:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a model locally**\n   - In a terminal, run:\n     ```bash\n     ollama run llama3\n     ```\n   - This will download the model, start it locally, and open a chat interface.\n\n3. **Check the local server**\n   - Ollama runs a server on `localhost:11434`. Test it with:\n     ```bash\n     curl http://localhost:11434\n     ```\n   - You should get a JSON response.\n\n4. **Use it from Python**\n   - Install the client:\n     ```bash\n     pip install ollama\n     ```\n   - Example:\n     ```python\n     import ollama\n\n     response = ollama.chat(\n         model=\'llama3\',\n         messages=[{"role": "user", "content": "Hello!"}]\n     )\n\n     print(res

In [27]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"course join discovered just discovered the course can I join enrollment enrollment open FAQ"}
iteration #2...
ASSISTANT:
Yes — you can join the course even if you just discovered it.

One important caveat: if you want a certificate, you need to submit your project while submissions are still open. If you’re only interested in learning, you can start right away.

If you want, I can also tell you whether you need to register or how the certificate/self-paced rules work.


'Yes — you can join the course even if you just discovered it.\n\nOne important caveat: if you want a certificate, you need to submit your project while submissions are still open. If you’re only interested in learning, you can start right away.\n\nIf you want, I can also tell you whether you need to register or how the certificate/self-paced rules work.'

In [28]:
agent_loop(instructions, "how many counties are there in kenya and what are their names?")

iteration #1...
function_call: search {"query":"how many counties are there in Kenya names counties Kenya"}
iteration #2...
function_call: search {"query":"Kenya counties 47 names official list"}
iteration #3...
ASSISTANT:
Kenya has **47 counties**.

Here are their names:

1. Baringo  
2. Bomet  
3. Bungoma  
4. Busia  
5. Elgeyo-Marakwet  
6. Embu  
7. Garissa  
8. Homa Bay  
9. Isiolo  
10. Kajiado  
11. Kakamega  
12. Kericho  
13. Kiambu  
14. Kilifi  
15. Kirinyaga  
16. Kisii  
17. Kisumu  
18. Kitui  
19. Kwale  
20. Laikipia  
21. Lamu  
22. Machakos  
23. Makueni  
24. Mandera  
25. Marsabit  
26. Meru  
27. Migori  
28. Mombasa  
29. Murang’a  
30. Nairobi City  
31. Nakuru  
32. Nandi  
33. Narok  
34. Nyamira  
35. Nyandarua  
36. Nyeri  
37. Samburu  
38. Siaya  
39. Taita-Taveta  
40. Tana River  
41. Tharaka-Nithi  
42. Trans Nzoia  
43. Turkana  
44. Uasin Gishu  
45. Vihiga  
46. Wajir  
47. West Pokot  

If you want, I can also group them by region or provide a map-st

'Kenya has **47 counties**.\n\nHere are their names:\n\n1. Baringo  \n2. Bomet  \n3. Bungoma  \n4. Busia  \n5. Elgeyo-Marakwet  \n6. Embu  \n7. Garissa  \n8. Homa Bay  \n9. Isiolo  \n10. Kajiado  \n11. Kakamega  \n12. Kericho  \n13. Kiambu  \n14. Kilifi  \n15. Kirinyaga  \n16. Kisii  \n17. Kisumu  \n18. Kitui  \n19. Kwale  \n20. Laikipia  \n21. Lamu  \n22. Machakos  \n23. Makueni  \n24. Mandera  \n25. Marsabit  \n26. Meru  \n27. Migori  \n28. Mombasa  \n29. Murang’a  \n30. Nairobi City  \n31. Nakuru  \n32. Nandi  \n33. Narok  \n34. Nyamira  \n35. Nyandarua  \n36. Nyeri  \n37. Samburu  \n38. Siaya  \n39. Taita-Taveta  \n40. Tana River  \n41. Tharaka-Nithi  \n42. Trans Nzoia  \n43. Turkana  \n44. Uasin Gishu  \n45. Vihiga  \n46. Wajir  \n47. West Pokot  \n\nIf you want, I can also group them by region or provide a map-style summary.'

In [29]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"Queen's Gambit chess opening"}
iteration #3...
function_call: search {"query":"queen gambit chess opening FAQ"}
iteration #4...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen gambit,” so it looks like this may be off-topic for the course.

If you meant something course-related, please rephrase with a course/logistics question, and I can help. Are there other areas you want to explore?


'I couldn’t find any course FAQ entry about “queen gambit,” so it looks like this may be off-topic for the course.\n\nIf you meant something course-related, please rephrase with a course/logistics question, and I can help. Are there other areas you want to explore?'

In [30]:
agent_loop(instructions, "how many counties are there in kenya and what are their names?")

iteration #1...
function_call: search {"query":"Kenya counties names how many counties are there in Kenya"}
iteration #2...
ASSISTANT:
This looks off-topic for the course FAQ, so I can’t answer it from the course materials.

If you have a course-related question, I’m happy to help. Are there other areas you want to explore?


'This looks off-topic for the course FAQ, so I can’t answer it from the course materials.\n\nIf you have a course-related question, I’m happy to help. Are there other areas you want to explore?'

In [31]:
agent_loop(instructions, "What is the latest date for registering for the data engineering course?")

iteration #1...
function_call: search {"query":"latest date registering data engineering course registration deadline"}
iteration #2...
function_call: search {"query":"data engineering course registration deadline latest date register"}
function_call: search {"query":"register data engineering course when can I still join latest date"}
function_call: search {"query":"data engineering course registration form open deadline"}
iteration #3...
ASSISTANT:
The FAQ doesn’t list a specific latest registration date for the data engineering course. What it does say is that registration is mainly to gauge interest, and you can still start learning and submitting homework while the form is open.

If you want, I can also help you find whether there’s a course-specific deadline for submissions or certificates.


'The FAQ doesn’t list a specific latest registration date for the data engineering course. What it does say is that registration is mainly to gauge interest, and you can still start learning and submitting homework while the form is open.\n\nIf you want, I can also help you find whether there’s a course-specific deadline for submissions or certificates.'